# The Value of Fine-Tuning LLMs
## Natural Language to SQL: Base Model vs. Fine-Tuned Model

This notebook demonstrates why fine-tuning matters. We'll build a data analysis agent that converts natural language questions into SQL queries against a **custom retail database**.

The database uses abbreviated, non-standard field names that a base model has never seen — so it will hallucinate column names. After fine-tuning on examples of correct SQL, the model learns the schema and generates accurate queries.

---
### Our Database Schema

| Table | Description | Key Fields |
|-------|-------------|------------|
| `ord_hdr` | Order headers | `ord_id`, `cust_ref`, `ord_dt`, `tot_amt`, `sts_cd` |
| `cust_mst` | Customer master | `cust_ref`, `cust_nm`, `rgn_cd`, `acq_src`, `acq_dt` |
| `prd_cat` | Product catalog | `prd_id`, `prd_nm`, `cat_cd`, `unit_prc`, `stk_qty` |
| `ord_dtl` | Order line items | `ord_id`, `prd_id`, `qty_ord`, `ln_amt` |

---

## Step 1: Setup

In [ ]:
# Install dependencies if needed
# !pip install openai

In [1]:
import getpass
import json
import time
import os
from openai import OpenAI

api_key = getpass.getpass("Enter your OpenAI API key: ")
client = OpenAI(api_key=api_key)
print("Client ready.")

Enter your OpenAI API key:  ········


Client ready.


## Step 2: Define the Schema and System Prompt

We give the model a description of the database. The base model has no prior knowledge of these table/column names.

In [2]:
SCHEMA_DESCRIPTION = """
You are a SQL expert for a retail company. Convert the user's natural language question into a SQL query.
Return ONLY the SQL query, no explanation.

Database schema:

Table: ord_hdr (order headers)
  - ord_id      : INTEGER PRIMARY KEY
  - cust_ref    : INTEGER (foreign key to cust_mst.cust_ref)
  - ord_dt      : DATE (order date)
  - tot_amt     : DECIMAL (total order amount)
  - sts_cd      : VARCHAR (status code: 'COMP'=completed, 'PEND'=pending, 'CNCL'=cancelled)

Table: cust_mst (customer master)
  - cust_ref    : INTEGER PRIMARY KEY
  - cust_nm     : VARCHAR (customer full name)
  - rgn_cd      : VARCHAR (region code: 'NA'=North America, 'EU'=Europe, 'APAC'=Asia Pacific)
  - acq_src     : VARCHAR (acquisition source: 'ORGANIC', 'PAID', 'REFERRAL')
  - acq_dt      : DATE (date customer was acquired)

Table: prd_cat (product catalog)
  - prd_id      : INTEGER PRIMARY KEY
  - prd_nm      : VARCHAR (product name)
  - cat_cd      : VARCHAR (category code)
  - unit_prc    : DECIMAL (unit price)
  - stk_qty     : INTEGER (stock quantity)

Table: ord_dtl (order line items)
  - ord_id      : INTEGER (foreign key to ord_hdr.ord_id)
  - prd_id      : INTEGER (foreign key to prd_cat.prd_id)
  - qty_ord     : INTEGER (quantity ordered)
  - ln_amt      : DECIMAL (line amount)
"""

## Step 3: Test the Base Model

Let's ask `gpt-4o-mini` to convert natural language to SQL **without** fine-tuning. Watch how it generates plausible-looking but wrong column names.

In [3]:
BASE_MODEL = "gpt-4o-mini"

def query_model(model_id, question):
    response = client.chat.completions.create(
        model=model_id,
        messages=[
            {"role": "system", "content": SCHEMA_DESCRIPTION},
            {"role": "user", "content": question}
        ],
        temperature=0
    )
    return response.choices[0].message.content.strip()


test_questions = [
    "Show me the top 5 customers by total order amount",
    "How many orders were completed in each region?",
    "Which products have less than 10 items in stock?",
    "What is the average order value for customers acquired through paid channels?",
]

print("=" * 60)
print(f"BASE MODEL: {BASE_MODEL}")
print("=" * 60)
for q in test_questions:
    sql = query_model(BASE_MODEL, q)
    print(f"\nQ: {q}")
    print(f"SQL:\n{sql}")
    print("-" * 40)

BASE MODEL: gpt-4o-mini

Q: Show me the top 5 customers by total order amount
SQL:
```sql
SELECT cust_ref, SUM(tot_amt) AS total_order_amount
FROM ord_hdr
GROUP BY cust_ref
ORDER BY total_order_amount DESC
LIMIT 5;
```
----------------------------------------

Q: How many orders were completed in each region?
SQL:
```sql
SELECT c.rgn_cd, COUNT(o.ord_id) AS completed_orders
FROM ord_hdr o
JOIN cust_mst c ON o.cust_ref = c.cust_ref
WHERE o.sts_cd = 'COMP'
GROUP BY c.rgn_cd;
```
----------------------------------------

Q: Which products have less than 10 items in stock?
SQL:
```sql
SELECT prd_id, prd_nm, stk_qty 
FROM prd_cat 
WHERE stk_qty < 10;
```
----------------------------------------

Q: What is the average order value for customers acquired through paid channels?
SQL:
```sql
SELECT AVG(tot_amt) AS average_order_value
FROM ord_hdr
WHERE cust_ref IN (
    SELECT cust_ref
    FROM cust_mst
    WHERE acq_src = 'PAID'
);
```
----------------------------------------


### Observations on the base model

Notice that the base model tends to:
- Use generic column names like `customer_id`, `total_amount`, `order_date` instead of `cust_ref`, `tot_amt`, `ord_dt`
- Sometimes guess table names incorrectly
- Join on wrong keys

This SQL **would not run** on our database. This is the problem fine-tuning solves.

---

## Step 4: Prepare the Fine-Tuning Dataset

We create 30 NL→SQL pairs using our exact schema. OpenAI fine-tuning requires JSONL format.

In [4]:
training_examples = [
    {
        "q": "Show all completed orders",
        "sql": "SELECT * FROM ord_hdr WHERE sts_cd = 'COMP';"
    },
    {
        "q": "List all customers from Europe",
        "sql": "SELECT * FROM cust_mst WHERE rgn_cd = 'EU';"
    },
    {
        "q": "Show me the top 5 customers by total order amount",
        "sql": "SELECT c.cust_ref, c.cust_nm, SUM(o.tot_amt) AS total_spent FROM cust_mst c JOIN ord_hdr o ON c.cust_ref = o.cust_ref GROUP BY c.cust_ref, c.cust_nm ORDER BY total_spent DESC LIMIT 5;"
    },
    {
        "q": "How many orders were completed in each region?",
        "sql": "SELECT c.rgn_cd, COUNT(o.ord_id) AS completed_orders FROM cust_mst c JOIN ord_hdr o ON c.cust_ref = o.cust_ref WHERE o.sts_cd = 'COMP' GROUP BY c.rgn_cd;"
    },
    {
        "q": "Which products have less than 10 items in stock?",
        "sql": "SELECT prd_id, prd_nm, stk_qty FROM prd_cat WHERE stk_qty < 10;"
    },
    {
        "q": "What is the average order value for customers acquired through paid channels?",
        "sql": "SELECT AVG(o.tot_amt) AS avg_order_value FROM ord_hdr o JOIN cust_mst c ON o.cust_ref = c.cust_ref WHERE c.acq_src = 'PAID';"
    },
    {
        "q": "Show all pending orders placed this year",
        "sql": "SELECT * FROM ord_hdr WHERE sts_cd = 'PEND' AND YEAR(ord_dt) = YEAR(CURDATE());"
    },
    {
        "q": "What products were ordered most frequently?",
        "sql": "SELECT p.prd_id, p.prd_nm, SUM(d.qty_ord) AS total_qty FROM prd_cat p JOIN ord_dtl d ON p.prd_id = d.prd_id GROUP BY p.prd_id, p.prd_nm ORDER BY total_qty DESC;"
    },
    {
        "q": "Find customers who have never placed an order",
        "sql": "SELECT c.cust_ref, c.cust_nm FROM cust_mst c LEFT JOIN ord_hdr o ON c.cust_ref = o.cust_ref WHERE o.ord_id IS NULL;"
    },
    {
        "q": "What is the total revenue from North America?",
        "sql": "SELECT SUM(o.tot_amt) AS total_revenue FROM ord_hdr o JOIN cust_mst c ON o.cust_ref = c.cust_ref WHERE c.rgn_cd = 'NA' AND o.sts_cd = 'COMP';"
    },
    {
        "q": "List all cancelled orders with customer names",
        "sql": "SELECT o.ord_id, c.cust_nm, o.ord_dt, o.tot_amt FROM ord_hdr o JOIN cust_mst c ON o.cust_ref = c.cust_ref WHERE o.sts_cd = 'CNCL';"
    },
    {
        "q": "How many customers were acquired organically per region?",
        "sql": "SELECT rgn_cd, COUNT(*) AS customer_count FROM cust_mst WHERE acq_src = 'ORGANIC' GROUP BY rgn_cd;"
    },
    {
        "q": "Show me the most expensive products",
        "sql": "SELECT prd_id, prd_nm, unit_prc FROM prd_cat ORDER BY unit_prc DESC LIMIT 10;"
    },
    {
        "q": "What is the total quantity ordered per product category?",
        "sql": "SELECT p.cat_cd, SUM(d.qty_ord) AS total_qty FROM prd_cat p JOIN ord_dtl d ON p.prd_id = d.prd_id GROUP BY p.cat_cd;"
    },
    {
        "q": "Which customers placed more than 5 orders?",
        "sql": "SELECT c.cust_ref, c.cust_nm, COUNT(o.ord_id) AS order_count FROM cust_mst c JOIN ord_hdr o ON c.cust_ref = o.cust_ref GROUP BY c.cust_ref, c.cust_nm HAVING order_count > 5;"
    },
    {
        "q": "Show daily order totals for the last 30 days",
        "sql": "SELECT ord_dt, SUM(tot_amt) AS daily_revenue FROM ord_hdr WHERE ord_dt >= DATE_SUB(CURDATE(), INTERVAL 30 DAY) GROUP BY ord_dt ORDER BY ord_dt;"
    },
    {
        "q": "Find the highest value order",
        "sql": "SELECT o.ord_id, c.cust_nm, o.ord_dt, o.tot_amt FROM ord_hdr o JOIN cust_mst c ON o.cust_ref = c.cust_ref ORDER BY o.tot_amt DESC LIMIT 1;"
    },
    {
        "q": "How many products are in each category?",
        "sql": "SELECT cat_cd, COUNT(*) AS product_count FROM prd_cat GROUP BY cat_cd;"
    },
    {
        "q": "Show me revenue by acquisition source",
        "sql": "SELECT c.acq_src, SUM(o.tot_amt) AS total_revenue FROM cust_mst c JOIN ord_hdr o ON c.cust_ref = o.cust_ref WHERE o.sts_cd = 'COMP' GROUP BY c.acq_src;"
    },
    {
        "q": "Which orders contain more than 3 different products?",
        "sql": "SELECT ord_id, COUNT(DISTINCT prd_id) AS product_count FROM ord_dtl GROUP BY ord_id HAVING product_count > 3;"
    },
    {
        "q": "Show all customers from APAC region acquired via referral",
        "sql": "SELECT cust_ref, cust_nm, acq_dt FROM cust_mst WHERE rgn_cd = 'APAC' AND acq_src = 'REFERRAL';"
    },
    {
        "q": "What is the average stock quantity per category?",
        "sql": "SELECT cat_cd, AVG(stk_qty) AS avg_stock FROM prd_cat GROUP BY cat_cd;"
    },
    {
        "q": "Show me orders with their line item details",
        "sql": "SELECT o.ord_id, o.ord_dt, o.sts_cd, d.prd_id, d.qty_ord, d.ln_amt FROM ord_hdr o JOIN ord_dtl d ON o.ord_id = d.ord_id;"
    },
    {
        "q": "Find the newest customers",
        "sql": "SELECT cust_ref, cust_nm, acq_dt, rgn_cd FROM cust_mst ORDER BY acq_dt DESC LIMIT 10;"
    },
    {
        "q": "What percentage of orders are pending?",
        "sql": "SELECT (COUNT(CASE WHEN sts_cd = 'PEND' THEN 1 END) * 100.0 / COUNT(*)) AS pct_pending FROM ord_hdr;"
    },
    {
        "q": "Show me the total line amount per order",
        "sql": "SELECT ord_id, SUM(ln_amt) AS total_line_amt FROM ord_dtl GROUP BY ord_id;"
    },
    {
        "q": "Which region has the most customers?",
        "sql": "SELECT rgn_cd, COUNT(*) AS customer_count FROM cust_mst GROUP BY rgn_cd ORDER BY customer_count DESC LIMIT 1;"
    },
    {
        "q": "Show products never ordered",
        "sql": "SELECT p.prd_id, p.prd_nm FROM prd_cat p LEFT JOIN ord_dtl d ON p.prd_id = d.prd_id WHERE d.prd_id IS NULL;"
    },
    {
        "q": "What is total revenue per month this year?",
        "sql": "SELECT MONTH(ord_dt) AS month, SUM(tot_amt) AS monthly_revenue FROM ord_hdr WHERE YEAR(ord_dt) = YEAR(CURDATE()) AND sts_cd = 'COMP' GROUP BY MONTH(ord_dt) ORDER BY month;"
    },
    {
        "q": "Find customers whose total spending exceeds 1000",
        "sql": "SELECT c.cust_ref, c.cust_nm, SUM(o.tot_amt) AS total_spent FROM cust_mst c JOIN ord_hdr o ON c.cust_ref = o.cust_ref WHERE o.sts_cd = 'COMP' GROUP BY c.cust_ref, c.cust_nm HAVING total_spent > 1000;"
    },
]

print(f"Training examples: {len(training_examples)}")

Training examples: 30


In [5]:
# Write training data to JSONL format required by OpenAI
training_file_path = "training_data.jsonl"

with open(training_file_path, "w") as f:
    for example in training_examples:
        record = {
            "messages": [
                {"role": "system", "content": SCHEMA_DESCRIPTION},
                {"role": "user", "content": example["q"]},
                {"role": "assistant", "content": example["sql"]}
            ]
        }
        f.write(json.dumps(record) + "\n")

print(f"Training file written: {training_file_path}")

# Preview one record
with open(training_file_path) as f:
    first = json.loads(f.readline())
print("\nSample record:")
print(json.dumps(first, indent=2))

Training file written: training_data.jsonl

Sample record:
{
  "messages": [
    {
      "role": "system",
      "content": "\nYou are a SQL expert for a retail company. Convert the user's natural language question into a SQL query.\nReturn ONLY the SQL query, no explanation.\n\nDatabase schema:\n\nTable: ord_hdr (order headers)\n  - ord_id      : INTEGER PRIMARY KEY\n  - cust_ref    : INTEGER (foreign key to cust_mst.cust_ref)\n  - ord_dt      : DATE (order date)\n  - tot_amt     : DECIMAL (total order amount)\n  - sts_cd      : VARCHAR (status code: 'COMP'=completed, 'PEND'=pending, 'CNCL'=cancelled)\n\nTable: cust_mst (customer master)\n  - cust_ref    : INTEGER PRIMARY KEY\n  - cust_nm     : VARCHAR (customer full name)\n  - rgn_cd      : VARCHAR (region code: 'NA'=North America, 'EU'=Europe, 'APAC'=Asia Pacific)\n  - acq_src     : VARCHAR (acquisition source: 'ORGANIC', 'PAID', 'REFERRAL')\n  - acq_dt      : DATE (date customer was acquired)\n\nTable: prd_cat (product catalog)\n  

## Step 5: Upload the Training File and Start Fine-Tuning

In [6]:
# Upload the training file to OpenAI
with open(training_file_path, "rb") as f:
    upload_response = client.files.create(file=f, purpose="fine-tune")

file_id = upload_response.id
print(f"File uploaded. ID: {file_id}")

File uploaded. ID: file-MH1w7sMrXULHRsXjrLjccF


In [7]:
# Start the fine-tuning job
ft_job = client.fine_tuning.jobs.create(
    training_file=file_id,
    model="gpt-4o-mini-2024-07-18",  # smallest, fastest, cheapest to fine-tune
    hyperparameters={"n_epochs": 3}
)

job_id = ft_job.id
print(f"Fine-tuning job started!")
print(f"Job ID : {job_id}")
print(f"Status : {ft_job.status}")
print("\nThis typically takes 5-15 minutes. Run the next cell to poll for completion.")

Fine-tuning job started!
Job ID : ftjob-wlJ6SqTX0kgOkOHjpJNYeEq5
Status : validating_files

This typically takes 5-15 minutes. Run the next cell to poll for completion.


## Step 6: Wait for Fine-Tuning to Complete

Poll every 30 seconds until the job finishes.

In [8]:
print("Polling for fine-tuning completion...")

while True:
    job = client.fine_tuning.jobs.retrieve(job_id)
    status = job.status
    print(f"[{time.strftime('%H:%M:%S')}] Status: {status}")

    if status == "succeeded":
        fine_tuned_model = job.fine_tuned_model
        print(f"\nFine-tuning complete!")
        print(f"Fine-tuned model: {fine_tuned_model}")
        break
    elif status in ("failed", "cancelled"):
        print(f"Job ended with status: {status}")
        print(job)
        break

    time.sleep(30)

Polling for fine-tuning completion...
[16:30:00] Status: validating_files
[16:30:30] Status: validating_files
[16:31:01] Status: queued
[16:31:31] Status: queued
[16:32:01] Status: queued
[16:32:32] Status: queued
[16:33:02] Status: queued
[16:33:32] Status: queued
[16:34:03] Status: queued
[16:34:33] Status: queued
[16:35:04] Status: queued
[16:35:34] Status: queued
[16:36:05] Status: queued
[16:36:35] Status: queued
[16:37:06] Status: queued
[16:37:36] Status: queued
[16:38:06] Status: queued
[16:38:37] Status: queued
[16:39:07] Status: queued
[16:39:38] Status: queued
[16:40:08] Status: queued
[16:40:39] Status: queued
[16:41:09] Status: queued
[16:41:39] Status: queued
[16:42:10] Status: queued
[16:42:40] Status: queued
[16:43:10] Status: queued
[16:43:41] Status: queued
[16:44:11] Status: queued
[16:44:41] Status: queued
[16:45:12] Status: queued
[16:45:42] Status: queued
[16:46:13] Status: queued
[16:46:43] Status: queued
[16:47:13] Status: queued
[16:47:44] Status: queued
[16:48

## Step 7: Test the Fine-Tuned Model

Now let's ask the **same questions** and compare outputs.

In [9]:
SCHEMA_DESCRIPTION_FT = """
You are a SQL expert for a retail company. Convert the user's natural language question into a SQL query.
Return ONLY the SQL query, no explanation.
"""

In [12]:
def query_model_ft(model_id, question):
    response = client.chat.completions.create(
        model=model_id,
        messages=[
            {"role": "system", "content": SCHEMA_DESCRIPTION},
            {"role": "user", "content": question}
        ],
        temperature=0
    )
    return response.choices[0].message.content.strip()

In [13]:
print("=" * 60)
print(f"FINE-TUNED MODEL: {fine_tuned_model}")
print("=" * 60)
for q in test_questions:
    sql = query_model_ft(fine_tuned_model, q)
    print(f"\nQ: {q}")
    print(f"SQL:\n{sql}")
    print("-" * 40)

FINE-TUNED MODEL: ft:gpt-4o-mini-2024-07-18:personal::Dpy4G5lK

Q: Show me the top 5 customers by total order amount
SQL:
SELECT c.cust_ref, c.cust_nm, SUM(o.tot_amt) AS total_spent FROM cust_mst c JOIN ord_hdr o ON c.cust_ref = o.cust_ref GROUP BY c.cust_ref, c.cust_nm ORDER BY total_spent DESC LIMIT 5;
----------------------------------------

Q: How many orders were completed in each region?
SQL:
SELECT c.rgn_cd, COUNT(o.ord_id) AS completed_orders FROM cust_mst c JOIN ord_hdr o ON c.cust_ref = o.cust_ref WHERE o.sts_cd = 'COMP' GROUP BY c.rgn_cd;
----------------------------------------

Q: Which products have less than 10 items in stock?
SQL:
SELECT prd_id, prd_nm, stk_qty FROM prd_cat WHERE stk_qty < 10;
----------------------------------------

Q: What is the average order value for customers acquired through paid channels?
SQL:
SELECT AVG(o.tot_amt) AS avg_order_value FROM ord_hdr o JOIN cust_mst c ON o.cust_ref = c.cust_ref WHERE c.acq_src = 'PAID';
---------------------------

## Step 8: Side-by-Side Comparison

In [14]:
comparison_questions = [
    "Show me the top 5 customers by total order amount",
    "How many orders were completed in each region?",
    "What is the average order value for customers acquired through paid channels?",
]

print("SIDE-BY-SIDE COMPARISON")
print("=" * 80)

for q in comparison_questions:
    base_sql = query_model(BASE_MODEL, q)
    ft_sql = query_model_ft(fine_tuned_model, q)

    print(f"\nQUESTION: {q}")
    print()
    print(f"  BASE MODEL ({BASE_MODEL}):")
    for line in base_sql.splitlines():
        print(f"    {line}")
    print()
    print(f"  FINE-TUNED MODEL ({fine_tuned_model}):")
    for line in ft_sql.splitlines():
        print(f"    {line}")
    print("\n" + "-" * 80)

SIDE-BY-SIDE COMPARISON

QUESTION: Show me the top 5 customers by total order amount

  BASE MODEL (gpt-4o-mini):
    ```sql
    SELECT cust_ref, SUM(tot_amt) AS total_order_amount
    FROM ord_hdr
    GROUP BY cust_ref
    ORDER BY total_order_amount DESC
    LIMIT 5;
    ```

  FINE-TUNED MODEL (ft:gpt-4o-mini-2024-07-18:personal::Dpy4G5lK):
    SELECT c.cust_ref, c.cust_nm, SUM(o.tot_amt) AS total_spent FROM cust_mst c JOIN ord_hdr o ON c.cust_ref = o.cust_ref GROUP BY c.cust_ref, c.cust_nm ORDER BY total_spent DESC LIMIT 5;

--------------------------------------------------------------------------------

QUESTION: How many orders were completed in each region?

  BASE MODEL (gpt-4o-mini):
    ```sql
    SELECT c.rgn_cd, COUNT(o.ord_id) AS completed_orders
    FROM ord_hdr o
    JOIN cust_mst c ON o.cust_ref = c.cust_ref
    WHERE o.sts_cd = 'COMP'
    GROUP BY c.rgn_cd;
    ```

  FINE-TUNED MODEL (ft:gpt-4o-mini-2024-07-18:personal::Dpy4G5lK):
    SELECT c.rgn_cd, COUNT(o.ord_id)

## Step 9: Interactive Agent

Try your own questions using the fine-tuned model.

In [ ]:
print("Data Analysis Agent (fine-tuned) — type 'quit' to exit\n")
print("Available tables: ord_hdr, cust_mst, prd_cat, ord_dtl")
print("-" * 60)

while True:
    question = input("\nYour question: ").strip()
    if question.lower() in ("quit", "exit", ""):
        print("Goodbye!")
        break
    sql = query_model_ft(fine_tuned_model, question)
    print(f"\nGenerated SQL:\n{sql}")

---
## Summary

| | Base Model | Fine-Tuned Model |
|---|---|---|
| **Column names** | Hallucinated (`customer_id`, `amount`) | Exact schema (`cust_ref`, `tot_amt`) |
| **Table names** | Generic guesses | Correct (`ord_hdr`, `cust_mst`) |
| **JOIN keys** | Wrong keys | Correct FK relationships |
| **Status codes** | Unknown values | Correct (`COMP`, `PEND`, `CNCL`) |
| **Runnable SQL** | No | Yes |

Fine-tuning with just **30 examples** was enough to teach the model the exact schema — making the difference between SQL that crashes and SQL that runs.